# 🎨 Google Colab AI Microservice - Virtual Try-On & Face Swap
### Đồ án Kiến trúc hướng dịch vụ (SOA) - Clothing Online

**Hướng dẫn 3 bước để chạy:**
1. Chọn menu **Runtime → Change runtime type → Chọn T4 GPU** (Miễn phí).
2. Điền `NGROK_TOKEN` của bạn ở Bước 4.
3. Bấm `Runtime → Run all` (hoặc `Ctrl + F9`). Copy link Ngrok sinh ra dán vào file `.env` ở máy tính.

### Bước 1: Cài đặt các thư viện cần thiết (Bao gồm Rembg AI tách nền)

In [ ]:
!pip install -q fastapi uvicorn pyngrok pydantic pillow opencv-python-headless numpy onnxruntime-gpu insightface rembg

### Bước 2: Tải Model Face Swapper (InsightFace inswapper_128.onnx)

In [ ]:
!wget -q -nc https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx -O inswapper_128.onnx
!ls -lh inswapper_128.onnx

### Bước 3: Khởi tạo mã nguồn FastAPI Server (Tách nền triệt để 100%)

In [ ]:
code = '''
import os, io, base64, time, cv2
import numpy as np
from PIL import Image
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional

app = FastAPI(title="Fashion AI Microservice (SOA)")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

def b64_to_pil(b64_str: str, mode: str = None) -> Image.Image:
    if ',' in b64_str: b64_str = b64_str.split(',')[1]
    img = Image.open(io.BytesIO(base64.b64decode(b64_str)))
    if mode: return img.convert(mode)
    return img

def pil_to_b64(pil_img: Image.Image, format: str = "PNG") -> str:
    buf = io.BytesIO()
    pil_img.save(buf, format=format)
    return f"data:image/{format.lower()};base64,{base64.b64encode(buf.getvalue()).decode('utf-8')}"

class FaceSwapRequest(BaseModel):
    base_image: str
    face_image: str

class TryOnRequest(BaseModel):
    model_image: str
    garment_image: str
    category: Optional[str] = "tops"

face_app = None
swapper = None
try:
    import insightface
    from insightface.app import FaceAnalysis
    face_app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
    face_app.prepare(ctx_id=0, det_size=(640, 640))
    if os.path.exists('inswapper_128.onnx'):
        swapper = insightface.model_zoo.get_model('inswapper_128.onnx', download=False, providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
        print("[+] InsightFace Swapper ready!")
except Exception as e:
    print(f"[!] Warning: {e}")

def extract_garment(garment_pil: Image.Image) -> Image.Image:
    try:
        from rembg import remove
        rgba = remove(garment_pil)
        bbox = rgba.getbbox()
        if bbox: return rgba.crop(bbox)
        return rgba
    except Exception:
        pass
    img = np.array(garment_pil)
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    corners = [img[0,0], img[0, w-1], img[h-1, 0], img[h-1, w-1]]
    bg_val = int(np.median([c[0] for c in corners]))
    threshold_val = max(220, bg_val - 12)
    _, binary = cv2.threshold(gray, threshold_val, 255, cv2.THRESH_BINARY_INV)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    clean_mask = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    clean_mask = cv2.morphologyEx(clean_mask, cv2.MORPH_CLOSE, kernel)
    clean_mask = cv2.GaussianBlur(clean_mask, (5, 5), 0)
    rgba = cv2.cvtColor(img, cv2.COLOR_RGB2RGBA)
    rgba[:, :, 3] = clean_mask
    pil_rgba = Image.fromarray(rgba)
    bbox = pil_rgba.getbbox()
    if bbox: pil_rgba = pil_rgba.crop(bbox)
    return pil_rgba

@app.get("/health")
def health():
    return {"status": "ok", "device": "cuda", "swapper": swapper is not None}

@app.post("/face-swap")
async def face_swap(req: FaceSwapRequest):
    base_pil = b64_to_pil(req.base_image)
    face_pil = b64_to_pil(req.face_image)
    base_np = np.array(base_pil)
    face_np = np.array(face_pil)
    if swapper is not None and face_app is not None:
        s_faces = face_app.get(cv2.cvtColor(face_np, cv2.COLOR_RGB2BGR))
        t_faces = face_app.get(cv2.cvtColor(base_np, cv2.COLOR_RGB2BGR))
        if len(s_faces) > 0 and len(t_faces) > 0:
            res_bgr = swapper.get(cv2.cvtColor(base_np, cv2.COLOR_RGB2BGR), t_faces[0], s_faces[0], paste_back=True)
            return {"image": pil_to_b64(Image.fromarray(cv2.cvtColor(res_bgr, cv2.COLOR_BGR2RGB)))}
    w_base, h_base = base_pil.size
    hr = int(w_base * 0.055)
    f_crop = face_pil.resize((hr * 2, int(hr * 2.4)), Image.Resampling.LANCZOS)
    mask = Image.new("L", f_crop.size, 0)
    mask_cv = cv2.ellipse(np.zeros(f_crop.size[::-1], dtype=np.uint8), (f_crop.size[0]//2, f_crop.size[1]//2), (f_crop.size[0]//2-2, f_crop.size[1]//2-2), 0, 0, 360, 255, -1)
    mask = Image.fromarray(cv2.GaussianBlur(mask_cv, (11, 11), 0))
    res = base_pil.copy()
    res.paste(f_crop, (int(w_base*0.5) - hr, int(h_base*0.14) - int(hr*1.2) + 5), mask)
    return {"image": pil_to_b64(res)}

@app.post("/try-on")
async def try_on(req: TryOnRequest):
    model_pil = b64_to_pil(req.model_image, mode="RGB")
    garment_pil = b64_to_pil(req.garment_image)
    category = req.category or "tops"
    mw, mh = model_pil.size
    if garment_pil.mode == "RGBA":
        bbox = garment_pil.getbbox()
        cropped = garment_pil.crop(bbox) if bbox else garment_pil
    else:
        cropped = extract_garment(garment_pil.convert("RGB"))
    gw, gh = cropped.size
    aspect = gh / max(1, gw)
    res = model_pil.copy()
    if category in ["tops", "shirt", "blazer", "tee", "knit", "tank"]:
        tw = int(mw * 0.48)
        th = int(tw * aspect)
        max_h = int(mh * 0.28)
        if th > max_h:
            th = max_h
            tw = int(th / aspect)
        px = int(mw * 0.5 - tw * 0.5)
        py = int(mh * 0.205)
    elif category in ["bottoms", "pants", "shorts", "skirt"]:
        tw = int(mw * 0.38)
        th = int(tw * aspect)
        max_h = int(mh * 0.45)
        if th > max_h:
            th = max_h
            tw = int(th / aspect)
        px = int(mw * 0.5 - tw * 0.5)
        py = int(mh * 0.40)
    else:
        tw = int(mw * 0.46)
        th = int(tw * aspect)
        max_h = int(mh * 0.56)
        if th > max_h:
            th = max_h
            tw = int(th / aspect)
        px = int(mw * 0.5 - tw * 0.5)
        py = int(mh * 0.205)
    resized = cropped.resize((tw, th), Image.Resampling.LANCZOS)
    res.paste(resized, (px, py), resized)
    return {"image": pil_to_b64(res)}
'''
with open('app.py', 'w') as f:
    f.write(code)
print('[+] app.py updated with Rembg zero-white-background!')

### Bước 4: Khởi chạy Server và Mở đường link Ngrok (Keep-Alive liên tục)

In [ ]:
import subprocess
import time
from pyngrok import ngrok

NGROK_TOKEN = ""  # <-- DIEN TOKEN NGROK CUA BAN VAO DAY

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

proc = subprocess.Popen(["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "5001"])
time.sleep(3)

public_url = ngrok.connect(5001).public_url
print("\n" + "="*60)
print("🎉 AI MICROSERVICE DA HOAT DONG THANH CONG!")
print(f"🔗 DUONG LINK AI CUA BAN: {public_url}")
print("="*60)
print(f"\n👉 Ban hay copy link tren va dan vao file .env o may tinh:\nAI_SERVICE_URL={public_url}")
print("="*60)
print("\n(Server dang chay de duy tri ket noi... Khong bam dung o nay!)")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    proc.terminate()
    print("Server da dung.")